### Language Translator: A Simple LLM Application with LCEL
In this quickstart we'll show you how to build a simple LLM application with LangChain. This application will translate text from English into another language. This is a relatively simple LLM application - it's just a single LLM call plus some prompting. Still, this is a great way to get started with LangChain - a lot of features can be built with just some prompting and an LLM call!

After seeing this video, you'll have a high level overview of:

- Using language models

- Using PromptTemplates and OutputParsers

- Using LangChain Expression Language (LCEL) to chain components together

- Debugging and tracing your application using LangSmith

- Deploying your application with LangServe

In [1]:
import os

from dotenv import load_dotenv

load_dotenv()

os.environ["OPENROUTER_API_KEY"]=os.getenv("OPENROUTER_API_KEY") or ""
os.environ["LANGSMITH_API_KEY"]=os.getenv("LANGSMITH_API_KEY") or ""
os.environ["LANGSMITH_TRACING"]=os.getenv("LANGSMITH_TRACING") or ""
os.environ["LANGSMITH_PROJECT"]=os.getenv("LANGSMITH_PROJECT") or ""

In [2]:
from langchain_openrouter import ChatOpenRouter

model = ChatOpenRouter(model="stepfun/step-3.5-flash:free")
model

d:\Agentic AI 101\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


ChatOpenRouter(profile={'max_input_tokens': 256000, 'max_output_tokens': 256000, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<openrouter.sdk.OpenRouter object at 0x000001DE6606E270>, openrouter_api_key=SecretStr('**********'), app_url='https://docs.langchain.com/oss', app_title='langchain', model_name='stepfun/step-3.5-flash:free', model_kwargs={})

In [3]:
from langchain_core.messages import HumanMessage, SystemMessage

messsages = [
    SystemMessage(content="Translate the following from English to French"),
    HumanMessage(content="Hello How are you?")
]

res = model.invoke(messsages)
res

AIMessage(content='Bonjour, comment allez-vous ?  \n\n*(Note: In informal contexts, "Comment ça va ?" or simply "Ça va ?" is also commonly used.)*', additional_kwargs={'reasoning_content': 'Okay, the user has asked to translate the English phrase "Hello How are you?" into French. This is a straightforward translation request with no complex context. \n\nThe phrase "Hello" can be translated as "Bonjour" which is the standard formal greeting. "How are you?" is typically "Comment allez-vous ?" in formal French, or "Ça va ?" in informal settings. Since the original English phrase is neutral, I\'ll use the formal version to be safe. \n\nI should provide both the translation and a brief note about formality since French greetings vary by context. The user didn\'t specify gender or region, so I\'ll keep it neutral. No need to overcomplicate—just the translation plus a short clarification.', 'reasoning_details': [{'type': 'reasoning.text', 'text': 'Okay, the user has asked to translate the Eng

In [4]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()
parser.invoke(res)

'Bonjour, comment allez-vous ?  \n\n*(Note: In informal contexts, "Comment ça va ?" or simply "Ça va ?" is also commonly used.)*'

Using LCEL to chain the components

In [5]:
chain = model | parser
chain.invoke(messsages)

'Bonjour, comment allez-vous ?'

Using Prompt templates

In [6]:
from langchain_core.prompts import ChatPromptTemplate

generic_template = "Translate the following into {language}"
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", generic_template),
        ("user", "{text}")
    ]
)

res = prompt.invoke({"language": "French", "text": "Hello"})
res.to_messages()

[SystemMessage(content='Translate the following into French', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Hello', additional_kwargs={}, response_metadata={})]

In [7]:
# Chaining together components with LCEL
chain = prompt | model | parser
chain.invoke({"language": "French", "text": "Hello"})

'Bonjour.'